In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

from sklearn.gaussian_process import GaussianProcessRegressor, GaussianProcessClassifier
from sklearn.gaussian_process.kernels import (
    RBF
)

from sklearn.neural_network import MLPRegressor, MLPClassifier


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


from sklearn.datasets import load_breast_cancer, make_moons

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("All imports successful!")

---

## Part 5: Classification — GP Confidence vs Neural Network Confidence

GPs work for classification too. Instead of predicting a number ± uncertainty, a GP classifier gives a **probability** for each class — and that probability is meaningful.

Neural networks *also* produce probabilities (via softmax/sigmoid), but those probabilities tend to be **overconfident**, especially far from training data.

Let's compare.

In [ ]:
# ============================================================
# 2D classification: confidence maps
# ============================================================
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)

# Train both models
gpc = GaussianProcessClassifier(kernel=RBF(1.0), random_state=42)
gpc.fit(X_moons, y_moons)

nn_clf = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=2000, random_state=42)
nn_clf.fit(X_moons, y_moons)

# Create prediction grid (including regions FAR from training data)
h = 0.05
xx, yy = np.meshgrid(np.arange(-3, 4, h), np.arange(-2, 3, h))
grid = np.c_[xx.ravel(), yy.ravel()]

# Get confidence (max probability) for each grid point
gp_probs = gpc.predict_proba(grid)
gp_conf = np.max(gp_probs, axis=1).reshape(xx.shape)

nn_probs = nn_clf.predict_proba(grid)
nn_conf = np.max(nn_probs, axis=1).reshape(xx.shape)

# Plot confidence maps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
cmap = ListedColormap(['#FF0000', '#0000FF'])

for ax, conf, title, color in [
    (axes[0], nn_conf, 'Neural Network Confidence', 'Reds'),
    (axes[1], gp_conf, 'Gaussian Process Confidence', 'Blues'),
]:
    im = ax.contourf(xx, yy, conf, levels=np.linspace(0.5, 1.0, 20), cmap=color, alpha=0.7)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap=cmap, 
              edgecolors='black', s=30, zorder=5)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    plt.colorbar(im, ax=ax, label='Confidence')

plt.suptitle('Where Is Each Model Confident?\n(Darker = more confident)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Look at the EDGES of each plot — far from any training data:")
print("  Neural network: DARK (highly confident) even in empty space")
print("  Gaussian process: LIGHT (low confidence) where there's no data")
print("\nThe GP's confidence fades to 50% (coin flip) far from training data.")
print("That's the correct behavior — it's saying 'I have no information here.'")

In [ ]:
# ============================================================
# The big issue: predictions on a far-away point
# ============================================================
far_points = np.array([
    [0.5, 0.25],  # near the center of training data
    [2, 1],   # outside but near training data
    [5.0, 5.0],   # completely outside training distribution
    [-3.0, -2.0],  # completely outside, other direction
])
far_labels = [
    'Center of data',
    'Just outside data',
    'Far away (5, 5)',
    'Far away (-3, -2)',
]

print("How confident is each model at different locations?")
print("=" * 65)
print(f"{'Location':<25s} {'NN Confidence':>15s} {'GP Confidence':>15s}")
print("-" * 65)

for point, label in zip(far_points, far_labels):
    nn_conf_pt = np.max(nn_clf.predict_proba(point.reshape(1, -1)))
    gp_conf_pt = np.max(gpc.predict_proba(point.reshape(1, -1)))
    print(f"{label:<25s} {nn_conf_pt:>14.1%} {gp_conf_pt:>14.1%}")

print("-" * 65)
print("\n At (5, 5) — completely outside the training data:")
print("   Neural network says: 'I'm ~100% sure it's class 1!'")
print("   Gaussian process says: 'I have no idea (50/50).'")

In [ ]:
# ============================================================
# Real data: Breast Cancer classification with GP
# ============================================================
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42, stratify=cancer.target
)

# Scale
scaler_c = StandardScaler()
X_train_cs = scaler_c.fit_transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

# GP classifier
gpc_cancer = GaussianProcessClassifier(kernel=RBF(1.0), random_state=42)
gpc_cancer.fit(X_train_cs, y_train_c)
gp_probs_c = gpc_cancer.predict_proba(X_test_cs)
gp_preds_c = gpc_cancer.predict(X_test_cs)
gp_conf_c = np.max(gp_probs_c, axis=1)

# NN classifier
nn_cancer = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
nn_cancer.fit(X_train_cs, y_train_c)
nn_probs_c = nn_cancer.predict_proba(X_test_cs)
nn_preds_c = nn_cancer.predict(X_test_cs)
nn_conf_c = np.max(nn_probs_c, axis=1)

print("Breast Cancer Classification Results:")
print(f"  GP Accuracy:  {accuracy_score(y_test_c, gp_preds_c):.4f}")
print(f"  NN Accuracy:  {accuracy_score(y_test_c, nn_preds_c):.4f}")

In [ ]:
# ============================================================
# Is the GP's confidence meaningful on real data?
# ============================================================
# Key test: when the model is WRONG, is it less confident?

gp_correct = gp_preds_c == y_test_c
nn_correct = nn_preds_c == y_test_c

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, conf, correct, name, color in [
    (axes[0], nn_conf_c, nn_correct, 'Neural Network', '#e74c3c'),
    (axes[1], gp_conf_c, gp_correct, 'Gaussian Process', 'steelblue'),
]:
    bins = np.linspace(0.5, 1.0, 20)
    ax.hist(conf[correct], bins=bins, alpha=0.6, label='Correct predictions', 
            color=color, edgecolor='black')
    ax.hist(conf[~correct], bins=bins, alpha=0.8, label='WRONG predictions', 
            color='gray', edgecolor='black')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10)
    
    # Stats
    mean_correct = conf[correct].mean()
    mean_wrong = conf[~correct].mean() if (~correct).sum() > 0 else 0
    ax.text(0.02, 0.80, f'Mean conf (correct): {mean_correct:.3f}\n'
            f'Mean conf (wrong):   {mean_wrong:.3f}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Confidence Distribution: Correct vs Wrong Predictions\n'
             '(A good model should be LESS confident when it\'s wrong)', 
             fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

print("A well-calibrated model should be LESS confident on its mistakes.")
print("The GP tends to express appropriate doubt when it gets things wrong.")
print("\nThis is enormously valuable in medical diagnosis: the model can flag")
print("cases where it's uncertain for human review, rather than confidently")
print("giving a wrong answer.")

---

## Part 6: Where Do GPs Struggle? 

If GPs are so great, why doesn't everyone use them? Because they have limitations.

In [ ]:
# ============================================================
# Limitation #1: GPs don't scale to large datasets
# ============================================================
import time

sizes = [50, 100, 200, 500, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]
gp_times = []
nn_times = []

for n in sizes:
    np.random.seed(42)
    X_timing = np.random.randn(n, 5)
    y_timing = np.random.randn(n)
    
    # Time GP
    t0 = time.time()
    gp_t = GaussianProcessRegressor(kernel=RBF(), random_state=42)
    gp_t.fit(X_timing, y_timing)
    gp_t.predict(X_timing[:10])
    gp_times.append(time.time() - t0)
    
    # Time NN
    t0 = time.time()
    nn_t = Pipeline([
        ('scaler', StandardScaler()),
        ('nn', MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=200, random_state=42))
    ])
    nn_t.fit(X_timing, y_timing)
    nn_t.predict(X_timing[:10])
    nn_times.append(time.time() - t0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, gp_times, 'o-', linewidth=2.5, markersize=8, 
        color='steelblue', label='Gaussian Process')
ax.plot(sizes, nn_times, 's-', linewidth=2.5, markersize=8, 
        color='#e74c3c', label='Neural Network')
ax.set_xlabel('Number of Training Samples')
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time: GP vs Neural Network', fontweight='bold', fontsize=13)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("GP training involves inverting an n×n matrix → O(n³) time complexity.")
print(f"  50 samples:   ~{gp_times[0]:.3f}s")
print(f"  10000 samples: ~{gp_times[-1]:.1f}s")
print(f"\nNeural networks scale much better to large datasets.")
print(f"For 10,000+ samples, GPs become impractical without approximations.")

### GP Strengths and Weaknesses

| Strength | Weakness |
|---|---|
| Built-in uncertainty quantification | Scales poorly to large datasets (O(n³)) |
| Uncertainty grows naturally away from data | Struggles in very high dimensions |
| Works well with small datasets | Kernel choice matters and requires thought |
| Few hyperparameters (kernel auto-tunes) | Can't easily learn hierarchical features |
| Mathematically principled | Assumes the kernel captures the true function's properties |

### What's Next: Bayesian Neural Networks

GPs give great uncertainty but can't handle big data. Neural networks handle big data but give no uncertainty. Is there a middle ground?

**Bayesian Neural Networks (BNNs)** attempt to combine both strengths:

| Approach | Prediction | Uncertainty | Scales? |
|---|---|---|---|
| Standard Neural Network | Single value |  None |  Yes |
| Gaussian Process | Mean + Std |  Calibrated |  O(n³) |
| Bayesian Neural Network | Mean + Std |  Approximate |  Mostly |

The idea: instead of learning a single weight value for each connection (like a standard NN), learn a **probability distribution** over weights. To predict, sample many sets of weights and see how much the predictions vary.

This is what I think the University of Michigan collaborators will introduce: taking the neural networks you've already built and making them Bayesian, so they can quantify uncertainty. Your project notebooks will serve as the testbed — comparing standard models (no uncertainty) against Bayesian approaches (with uncertainty) on the same data.

---

## Summary


| Concept | Key Idea |
|---|---|
| **Uncertainty Quantification (UQ)** | Predictions without error bars are incomplete — especially for engineering and safety-critical applications |
| **Point predictions** | Standard ML models (NN, SVM, trees) give a single number with no confidence measure |
| **Gaussian Process** | A model that predicts a distribution (mean ± std), not just a point |
| **Kernel function** | Defines similarity between inputs — controls GP smoothness and behavior |
| **Uncertainty grows with distance** | GP is confident near training data, uncertain far away — exactly what you want |
| **Calibration** | A model's confidence should match its actual accuracy |
| **GP limitations** | Scales O(n³), struggles in high dimensions — motivates Bayesian neural networks |

### The Big Picture

You now have a complete progression:

1. **Simple models** (linear/logistic regression): Interpretable, limited capacity
2. **Flexible models** (trees, forests, SVMs): More powerful, still point predictions
3. **Neural networks**: Very powerful, still point predictions
4. **Gaussian processes**: Less scalable, but gives you uncertainty for free
5. **Bayesian neural networks** : The best of both worlds — flexibility AND uncertainty

The progression isn't just "newer = better." Each tool has its place. The key insight is that **knowing what you don't know** is just as important as making good predictions — and for many engineering problems, it's more important.

### A Quick Introduction to Bayesian Thinking

Before the Michigan collaborators present, let's briefly introduce the **Bayesian** approach — a different way of thinking about learning from data that makes uncertainty quantification natural.

#### The Standard (Frequentist) Approach — What You've Done So Far

Every model you've built works like this: find the **single best set of parameters** that fits the training data. A linear regression finds one set of coefficients. A neural network finds one set of weights. The result is a single model that makes a single prediction.

#### The Bayesian Approach — A Different Philosophy

The Bayesian approach says: instead of finding one "best" answer, consider **all possible answers** and how plausible each one is.

It follows three steps:

1. **Prior:** Before seeing any data, what do you believe? Maybe you think the weights are probably small numbers near zero, but you're not sure. This is your starting assumption — your "prior belief."

2. **Evidence:** You observe training data. Some parameter values explain the data well; others don't.

3. **Posterior:** Combine your prior belief with the evidence to get an **updated belief** — a probability distribution over all possible parameter values. Parameters that explain the data well get higher probability. Parameters that don't get lower probability.

The key insight: **you don't end up with one answer. You end up with a distribution of answers.** And the spread of that distribution IS your uncertainty.

#### A Concrete Analogy

Suppose a friend tells you they ran a race. Before they tell you their time:

- **Prior:** You know they're a casual jogger, so you guess somewhere around 25-35 minutes for a 5K. That's your prior.
- **Evidence:** They mention they've been training hard for 6 months. This shifts your belief — maybe 20-28 minutes.
- **More evidence:** They say the course was hilly. You shift again — maybe 23-32 minutes.

At each step, you don't have a single number. You have a **range** that reflects your uncertainty. More evidence narrows the range.

This is exactly what Bayesian ML does with model parameters.

#### Applied to Neural Networks

A standard neural network learns that weight $w_3$ should be 0.72. That's it. One number.

A **Bayesian neural network** learns that weight $w_3$ is probably around 0.72, but could plausibly be anywhere from 0.5 to 0.9. When you predict, you sample many possible values of $w_3$ (and all other weights), run the network each time, and see how much the predictions vary.

- Near training data: all plausible weight values give similar predictions → **low uncertainty**
- Far from training data: different weight values give wildly different predictions → **high uncertainty**

Sound familiar? That's exactly how the GP behaved — and it's not a coincidence.

In [ ]:
# ============================================================
# Quick demo: the Bayesian idea using an ensemble of networks
# ============================================================
# We can approximate Bayesian uncertainty with a simple trick:
# train MANY neural networks (different random initializations)
# and use their DISAGREEMENT as a measure of uncertainty.
#
# This is actually a real technique called "deep ensembles," and it's
# one of the simplest ways to get uncertainty from neural networks.

# ============================================================
# Create data with a deliberate gap
# ============================================================
np.random.seed(42)

# Training data: observations on the LEFT and RIGHT, but nothing in the MIDDLE
X_left = np.random.uniform(0, 3, 25).reshape(-1, 1)
X_right = np.random.uniform(7, 10, 25).reshape(-1, 1)
X_train = np.vstack([X_left, X_right])

# True function: a sine wave with some noise
y_train = np.sin(X_train.ravel()) + 0.3 * np.random.randn(len(X_train))

# Where we'll ask for predictions (including the gap!)
X_plot = np.linspace(-1, 12, 300).reshape(-1, 1)
y_true = np.sin(X_plot.ravel())  # the true underlying function

# Gaussian Process
# ============================================================
kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10,  # try multiple starting points for kernel optimization
    random_state=42
)
gp.fit(X_train, y_train)

# The key: predict returns both mean and uncertainty
y_mean, y_std = gp.predict(X_plot, return_std=True)


# Train 40 neural networks on our gap dataset
ensemble_preds = []
for seed in range(40):
    nn_i = Pipeline([
        ('scaler', StandardScaler()),
        ('nn', MLPRegressor(hidden_layer_sizes=(50, 25), activation='relu',
                             max_iter=2000, random_state=seed))
    ])
    nn_i.fit(X_train, y_train)
    ensemble_preds.append(nn_i.predict(X_plot))

ensemble_preds = np.array(ensemble_preds)  # shape: (20, 300)
ens_mean = ensemble_preds.mean(axis=0)
ens_std = ensemble_preds.std(axis=0)

# Compare: GP uncertainty vs ensemble uncertainty
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Ensemble NN ---
ax = axes[0]
ax.fill_between(X_plot.ravel(), ens_mean - 2*ens_std, ens_mean + 2*ens_std,
                alpha=0.15, color='#e74c3c')
ax.fill_between(X_plot.ravel(), ens_mean - ens_std, ens_mean + ens_std,
                alpha=0.25, color='#e74c3c')
ax.plot(X_plot, ens_mean, '#e74c3c', linewidth=2.5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.scatter(X_train, y_train, c='black', s=40, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.set_title('NN Ensemble (20 networks)\n"Poor Man\'s Bayesian"', 
             fontweight='bold', fontsize=13, color='#e74c3c')

# --- Gaussian Process ---
ax = axes[1]
ax.fill_between(X_plot.ravel(), y_mean - 2*y_std, y_mean + 2*y_std,
                alpha=0.15, color='steelblue')
ax.fill_between(X_plot.ravel(), y_mean - y_std, y_mean + y_std,
                alpha=0.25, color='steelblue')
ax.plot(X_plot, y_mean, 'steelblue', linewidth=2.5)
ax.plot(X_plot, y_true, 'k--', alpha=0.3)
ax.scatter(X_train, y_train, c='black', s=40, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-3, 3)
ax.set_title('Gaussian Process\n(Principled Bayesian)', 
             fontweight='bold', fontsize=13, color='steelblue')

plt.suptitle('Two Ways to Get Uncertainty: Ensemble vs GP', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Both approaches show wider bands in the gap and at the edges!")
print("The ensemble is a rough approximation — it captures the idea that")
print("'different plausible models disagree here, so we're uncertain.'")
print("")
print("The GP does this rigorously through Bayesian math.")
print("Bayesian Neural Networks (what U of M will cover) do it more")
print("rigorously than ensembles but more scalably than GPs — by putting")
print("probability distributions directly on the network weights.")